# Материалы учебного проекта

Пути адаптированы к структуре репозитория; пустые ячейки удалены. Расчётная логика сохранена. Исторические выводы могут отражать промежуточные запуски. Перепроверенные показатели находятся в README, ограничения — в supporting/methodology_notes.md.


# 📊 Анализ рынка смесителей — Очистка и обогащение данных

In [1]:
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/faucets_sales_2023_2024.xlsx').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from the extracted repository folder.')
RAW_FILE = PROJECT_ROOT / 'data/raw/faucets_sales_2023_2024.xlsx'
DATA_OUTPUT = PROJECT_ROOT / 'data/processed'
FIGURE_OUTPUT = PROJECT_ROOT / 'reports/generated'
DATA_OUTPUT.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 1. Загрузка данных
df = pd.read_excel(RAW_FILE)
df_clean = df.copy()
print(f"Загружено строк: {len(df_clean):,}, колонок: {len(df_clean.columns)}")
print("Колонки:", df_clean.columns.tolist())
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.family']       = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', palette='muted')


Загружено строк: 222,615, колонок: 10
Колонки: ['brand_2', 'brand', 'color', 'month', 'cat1', 'cat2', 'cat3', 'article_name', 'revenue', 'quantity']


 ## 1. Первичный осмотр данных

In [2]:
print("=== Типы данных и заполненность ===")
info = pd.DataFrame({
    'dtype': df_clean.dtypes,
    'non_null': df_clean.notna().sum(),
    'null': df_clean.isna().sum(),
    'null_%': (df_clean.isna().sum() / len(df_clean) * 100).round(1),
    'unique': df_clean.nunique()
})
display(info)
print()
print("=== Числовая статистика ===")
display(df_clean[['revenue','quantity']].describe().map(lambda x: f'{x:,.0f}'))


=== Типы данных и заполненность ===


,dtype,non_null,null,null_%,unique
brand_2,object,222607,8,0.0,431
brand,object,222607,8,0.0,1180
color,object,222563,52,0.0,20
month,datetime64[ns],222615,0,0.0,24
cat1,object,222615,0,0.0,1
cat2,object,222615,0,0.0,1
cat3,object,52616,169999,76.4,2
article_name,object,222615,0,0.0,53229
revenue,int64,222615,0,0.0,56194
quantity,int64,222615,0,0.0,585



=== Числовая статистика ===


,revenue,quantity
count,"222,615","222,615"
mean,"26,193",7
std,"77,767",32
min,"-133,792",-8
25%,"3,848",1
50%,"8,536",1
75%,"20,275",4
max,"3,386,525","1,816"


## 2. Колонка `month` — тип данных

In [3]:
# month уже в формате datetime после загрузки через pd.read_excel
# Для надёжности принудительно приводим к datetime
df_clean['month'] = pd.to_datetime(df_clean['month'], errors='coerce')

# Вспомогательные колонки для анализа
df_clean['year']    = df_clean['month'].dt.year
df_clean['month_n'] = df_clean['month'].dt.month   # номер месяца 1-12
df_clean['quarter'] = df_clean['month'].dt.to_period('Q').astype(str)

## 3. Колонки `cat1`, `cat2` — проверка

In [4]:
print("cat1 (уникальные):", df_clean['cat1'].unique())
print("cat2 (уникальные):", df_clean['cat2'].unique())
print()
print("Обе колонки содержат только одно значение — они не несут")
print("аналитической ценности в текущем датасете (все записи — кухонные смесители).")
print("Оставляем как есть, используем cat2 только для контекста.")

cat1 (уникальные): ['Смесители']
cat2 (уникальные): ['Для кухни']

Обе колонки содержат только одно значение — они не несут
аналитической ценности в текущем датасете (все записи — кухонные смесители).
Оставляем как есть, используем cat2 только для контекста.


## 4. Колонка `color` — нормализация

**Проблема:** ~47% строк имеют значение `0` вместо пустого.
Ещё 52 строки — настоящий `NaN`. Плюс часть строк, где color ≠ реальный цвет (материал ≠ цвет).

**Решение:** парсим цвет из `article_name` и используем его как основной источник истины.

In [120]:
import re
import numpy as np

# Исправляем опечатку в исходных данных
before = (df_clean["color"] == "оружейная/вороненаясталь").sum()
df_clean["color"] = df_clean["color"].replace("оружейная/вороненаясталь", "оружейная/вороненая сталь")
after = (df_clean["color"] == "оружейная/вороненаясталь").sum()
print(f"Исправлено строк: {before - after}")

# Функция для парсинга ТОЛЬКО цвета (без металлов)
def parse_color_only(text):
    if pd.isna(text):
        return np.nan

    text = str(text).lower()
    if not text:
        return np.nan

    # Все цвета (включая шампань и бежевый)
    colors = {
        'черн(?:ый|ая|ое|ые|ого|ому|ую|ым|ом)': 'черный',
        'чёрн(?:ый|ая|ое|ые|ого|omu|ую|ым|ом)': 'черный',
        'бел(?:ый|ая|ое|ые|ого|ому|ую|ым|ом)': 'белый',
        'красн(?:ый|ая|ое|ые|ого|ому|ую|ым|ом)': 'красный',
        'син(?:ий|яя|ее|ие|его|ему|юю|им|ем)': 'синий',
        'зелен(?:ый|ая|ое|ые|ого|ому|ую|ым|ом)': 'зеленый',
        'желт(?:ый|ая|ое|ые|ого|ому|ую|ым|ом)': 'желтый',
        'жёлт(?:ый|ая|ое|ые|ого|ому|ую|ым|ом)': 'желтый',
        'коричнев(?:ый|ая|ое|ые|ого|ому|ую|ым|ом)': 'коричневый',
        'розов(?:ый|ая|ое|ые|ого|ому|ую|ым|ом)': 'розовый',
        'фиолетов(?:ый|ая|ое|ые|ого|ому|ую|ым|ом)': 'фиолетовый',
        'оранжев(?:ый|ая|ое|ые|ого|ому|ую|ым|ом)': 'оранжевый',
        'бежев(?:ый|ая|ое|ые)': 'бежевый',
        'молочн(?:ый|ая|ое|ые)': 'бежевый',
        'песочн(?:ый|ая|ое|ые)': 'песочный',
        'терракотов(?:ый|ая|ое|ые)': 'терракотовый',
        'шампан(?:ь|евый|евая|евое|евые)': 'шампань'  # шампань как цвет
    }

    for pattern, color in colors.items():
        if re.search(pattern, text):
            return color

    # Серый - это цвет
    if re.search(r'сер(?:ый|ая|ое|ые)', text):
        return 'серый'

    return np.nan

# Функция для парсинга ТОЛЬКО металлов/покрытий
def parse_metal_only(text):
    if pd.isna(text):
        return np.nan

    text = str(text).lower()
    if not text:
        return np.nan

    # Хром и серебро
    if re.search(r'хром(?:овый|овая|овое|овые)?', text):
        return 'хром'

    if re.search(r'серебр(?:о|янн(?:ый|ая|ое|ые)|ист(?:ый|ая|ое|ые))', text):
        return 'серебро'

    # Сталь (все виды стали объединяем в оружейную/вороненую сталь)
    if re.search(r'оружейн(?:ая|ой|ую|ые|ых)\s+стал[иь]|воронен(?:ая|ой|ую|ые|ых)\s+стал[иь]|вулканит|стал[иь]|нержавеющ(?:ая|ей|ую|ие|их)|нержавейк[аи]', text):
        return 'оружейная/вороненая сталь'

    # Материалы
    materials = {
        'бронз(?:а|вый|вая|вое|вые)': 'бронза',
        'золот(?:о|ой|ая|ое|ые)': 'золото',
        'латун(?:ь|ный|ная|ное|ные)': 'латунь',
        'мед(?:ь|ный|ная|ное|ные)': 'медь',
        'графит(?:овый|овая|овое|овые)?': 'графит',
        'сатин(?:овый|овая|овое|овые)?': 'сатин/никель',  # теперь только сатин/никель
        'никел(?:ь|евый|евая|евое|евые)': 'сатин/никель',  # тоже сатин/никель
    }

    for pattern, metal in materials.items():
        if re.search(pattern, text):
            return metal

    return np.nan

# Функция для определения уценки
def parse_discount(text):
    if pd.isna(text):
        return 'нет'

    text = str(text).lower()
    keywords = ['уценк', 'уценен', 'дефект', 'вскрыт', 'б/у', 'б у ', 'восстановл',
                'damaged', 'defect', 'used', 'second hand', 'open box']

    return 'да' if any(k in text for k in keywords) else 'нет'

# Функция для создания финальных колонок цвета и металла
def create_color_and_metal(row):
    # Получаем значение из color, если есть
    if pd.notna(row['color']) and row['color'] != 0 and row['color'] != '0' and str(row['color']).strip() != '':
        color_clean = str(row['color']).strip()
    else:
        color_clean = np.nan

    # Парсим только цвет из названия
    color_only = parse_color_only(row['article_name'])

    # Парсим только металл из названия
    metal_only = parse_metal_only(row['article_name'])

    # Определяем финальный цвет
    final_color = np.nan
    final_metal = np.nan

    # Приоритет для цвета: сначала проверяем color_clean, потом название
    if pd.notna(color_clean):
        color_from_clean = parse_color_only(color_clean)
        if pd.notna(color_from_clean):
            final_color = color_from_clean
    if pd.isna(final_color) and pd.notna(color_only):
        final_color = color_only

    # Приоритет для металла: сначала проверяем color_clean, потом название
    if pd.notna(color_clean):
        metal_from_clean = parse_metal_only(color_clean)
        if pd.notna(metal_from_clean):
            final_metal = metal_from_clean
    if pd.isna(final_metal) and pd.notna(metal_only):
        final_metal = metal_only

    return pd.Series({'color_final': final_color, 'metal_final': final_metal})

# Создаем финальные колонки
color_metal = df_clean.apply(create_color_and_metal, axis=1)
df_clean['color_final'] = color_metal['color_final']
df_clean['metal_final'] = color_metal['metal_final']

# Добавляем признак уценки
df_clean['is_discounted'] = df_clean['article_name'].apply(parse_discount)

# Проверяем результат
print("\n=== РАСПРЕДЕЛЕНИЕ color_final ===")
print(df_clean['color_final'].value_counts(dropna=False))

print("\n=== РАСПРЕДЕЛЕНИЕ metal_final ===")
print(df_clean['metal_final'].value_counts(dropna=False))

print("\n=== РАСПРЕДЕЛЕНИЕ is_discounted ===")
print(df_clean['is_discounted'].value_counts())

# Проверка комбинаций
print("\n=== ТОП-10 КОМБИНАЦИЙ ЦВЕТ + МЕТАЛЛ ===")
combinations = df_clean.dropna(subset=['color_final', 'metal_final']).groupby(['color_final', 'metal_final']).size().reset_index(name='count')
combinations['percent'] = (combinations['count'] / len(df_clean) * 100).round(2)
combinations = combinations.sort_values('count', ascending=False).head(10)
for i, row in combinations.iterrows():
    print(f"{row['color_final']} + {row['metal_final']}\t{row['count']}\t{row['percent']}%")

print("\n=== ПРОВЕРКА: латунь (должна быть в metal_final) ===")
latun_examples = df_clean[df_clean['article_name'].str.contains('латунь', case=False, na=True)][['color_final', 'metal_final', 'article_name']].head(10)
for _, row in latun_examples.iterrows():
    print(f"  color: {row['color_final']:20} | metal: {row['metal_final']:20} | article: {row['article_name'][:60]}...")

print("\n=== ПРОВЕРКА: хром (должен быть в metal_final) ===")
chrome_examples = df_clean[df_clean['article_name'].str.contains('хром', case=False, na=True)][['color_final', 'metal_final', 'article_name']].head(10)
for _, row in chrome_examples.iterrows():
    print(f"  color: {row['color_final']:20} | metal: {row['metal_final']:20} | article: {row['article_name'][:60]}...")

print("\n=== ПРОВЕРКА: сатин/никель (должно быть в metal_final без шампань/бежевый) ===")
satin_examples = df_clean[df_clean['article_name'].str.contains('сатин|никель', case=False, na=True)][['color_final', 'metal_final', 'article_name']].head(10)
for _, row in satin_examples.iterrows():
    print(f"  color: {row['color_final']:20} | metal: {row['metal_final']:20} | article: {row['article_name'][:60]}...")

print("\n=== ПРОВЕРКА: шампань (теперь должна быть в color_final) ===")
shampagne_examples = df_clean[df_clean['article_name'].str.contains('шампан', case=False, na=True)][['color_final', 'metal_final', 'article_name']].head(10)
for _, row in shampagne_examples.iterrows():
    print(f"  color: {row['color_final']:20} | metal: {row['metal_final']:20} | article: {row['article_name'][:60]}...")

print("\n=== ПРОВЕРКА: уценка ===")
discount_examples = df_clean[df_clean['is_discounted'] == 'да'][['is_discounted', 'article_name']].head(10)
for _, row in discount_examples.iterrows():
    print(f"  is_discounted: {row['is_discounted']} | article: {row['article_name'][:70]}...")

Исправлено строк: 0

=== РАСПРЕДЕЛЕНИЕ color_final ===
color_final
NaN             97248
серый           59544
черный          36837
белый           12234
бежевый          8421
песочный         2063
зеленый           827
красный           807
коричневый        580
синий             475
терракотовый      371
оранжевый         355
желтый            318
шампань           295
фиолетовый        158
розовый           133
Name: count, dtype: int64

=== РАСПРЕДЕЛЕНИЕ metal_final ===
metal_final
NaN                          108411
хром                          56280
оружейная/вороненая сталь     19359
латунь                        13781
сатин/никель                   7824
золото                         4875
бронза                         4321
графит                         3508
серебро                        2066
медь                            241
Name: count, dtype: int64

=== РАСПРЕДЕЛЕНИЕ is_discounted ===
is_discounted
нет    216271
да       4395
Name: count, dtype: int64

=== ТОП-10 КОМБИ

In [121]:
df_clean

,brand_2,brand,color,month,cat1,cat2,cat3,article_name,revenue,quantity,...,product_type,has_pullout,brand_abc,has_eco_aerator,market_avg_price,price_index,brand_active_months,brand_maturity,revenue_per_sku,qty_per_sku
0,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-07-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,23010,1,...,однорычажный,1,A,0,5693.210952,4.042,24.0,зрелый (>12 мес),64513.0,2.7
1,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-09-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,25560,1,...,однорычажный,1,A,0,5896.427217,4.335,24.0,зрелый (>12 мес),64513.0,2.7
2,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-10-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,25560,1,...,однорычажный,1,A,0,5899.046808,4.333,24.0,зрелый (>12 мес),64513.0,2.7
3,Hansgrohe,Hansgrohe,0,2023-09-01,Смесители,Для кухни,NaN,Hansgrohe Zesis M33 Кухонный смеситель однорыч...,17490,1,...,однорычажный,0,A,0,5896.427217,2.966,24.0,зрелый (>12 мес),64513.0,2.7
4,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-06-01,Смесители,Для кухни,NaN,Hansgrohe Focus M42 Кухонный смеситель однорыч...,33250,1,...,однорычажный,1,A,0,5631.260014,5.905,24.0,зрелый (>12 мес),64513.0,2.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220661,Zein,Zein,хром/серебро/сталь/серый,2023-06-01,Смесители,Для кухни,NaN,"Смеситель для кухни ZEIN ZC2034, высокий корпу...",6110,2,...,не определен,0,C,0,5631.260014,0.543,24.0,зрелый (>12 мес),19071.0,13.6
220662,Dokorona,Dokorona,0,2024-01-01,Смесители,Для кухни,NaN,Смеситель кухонный с гибким изливом (гусаком) ...,2893,1,...,не определен,0,C,0,5859.589131,0.494,21.0,зрелый (>12 мес),7707.0,4.4
220663,Нет бренда,Нет бренда,черный,2023-05-01,Смесители,Для кухни,NaN,"Смеситель для кухни, Кран однорычажный стально...",1194,1,...,однорычажный,0,A,0,5563.708806,0.215,24.0,зрелый (>12 мес),145531.0,50.7
220664,Нет бренда,Нет бренда,черный,2023-06-01,Смесители,Для кухни,NaN,"Смеситель для кухни, Кран однорычажный стально...",1194,1,...,однорычажный,0,A,0,5631.260014,0.212,24.0,зрелый (>12 мес),145531.0,50.7


## 5. Колонка `cat3` — заполнение пропусков

**Проблема:** 169 999 строк (76%) без значения — NaN.

**Решение:** ищем ключевые слова фильтра в `article_name`.

In [122]:
filter_patterns = [
    r'с фильтром', r'под фильтр', r'для фильтра',
    r'с подключением к? фильтру?', r'с подключением к? питьевой воде',
    r'питьевой воды?', r'очистки воды', r'питьевая вода',
    r'краник? для фильтра', r'кран для фильтра',
    r'с системой фильтрации', r'фильтрацией', r'фильтрации',
    r'with filter', r'water filter', r'filtration',
]

def parse_filter(text):
    if pd.isna(text): return 'Без фильтра'
    text = str(text).lower()
    return 'С фильтром' if any(re.search(p, text) for p in filter_patterns) else 'Без фильтра'

def create_cat3_new(row):
    # Явно указанное 'С фильтром' оставляем
    if pd.notna(row['cat3']) and str(row['cat3']).strip() == 'С фильтром':
        return 'С фильтром'
    return parse_filter(row['article_name'])

df_clean['cat3_new'] = df_clean.apply(create_cat3_new, axis=1)

print("=== РАСПРЕДЕЛЕНИЕ cat3_new ===")
print(df_clean['cat3_new'].value_counts())
print()
print("Было NaN в cat3:", df_clean['cat3'].isna().sum())
print("Новых 'С фильтром' (найдено из article_name):",
      ((df_clean['cat3'].isna()) & (df_clean['cat3_new'] == 'С фильтром')).sum())

=== РАСПРЕДЕЛЕНИЕ cat3_new ===
cat3_new
Без фильтра    164732
С фильтром      55934
Name: count, dtype: int64

Было NaN в cat3: 168494
Новых 'С фильтром' (найдено из article_name): 7112


## 6. Колонки `brand` / `brand_2` — консолидация

**Проблема:** `brand_2 == 'Прочие бренды'` — агрегированная категория для небольших брендов,
тогда как `brand` содержит реальное название.
`brand == 'Нет бренда'` — безымянные товары.

**Решение:**
- Если `brand_2 == 'Прочие бренды'` → оставляем значение из `brand` (реальный бренд)
- Если `brand == 'Нет бренда'` → `NoName/Прочие`
- `brand_2` больше не нужен для анализа (оставляем в df_clean как исходное поле)

In [123]:

def fix_brand(row):
    b  = str(row['brand']).strip()  if pd.notna(row['brand'])   else ''
    b2 = str(row['brand_2']).strip() if pd.notna(row['brand_2']) else ''

    # Если brand_2 = 'Прочие бренды' — реальный бренд в поле brand
    if b2 == 'Прочие бренды' and b not in ['', 'nan', 'Нет бренда']:
        return b

    # Безымянные товары
    if b in ['Нет бренда', '', 'nan'] or pd.isna(row['brand']):
        return 'NoName/Прочие'

    return b

df_clean['brand_clean'] = df_clean.apply(fix_brand, axis=1)

print(f"Уникальных брендов после очистки: {df_clean['brand_clean'].nunique()}")
print()
print("ТОП-20 брендов по выручке:")
top20 = (df_clean.groupby('brand_clean')['revenue']
         .sum().sort_values(ascending=False).head(20)
         .reset_index())
top20['revenue_млн'] = (top20['revenue'] / 1e6).round(1)
display(top20[['brand_clean', 'revenue_млн']])

Уникальных брендов после очистки: 1180

ТОП-20 брендов по выручке:


,brand_clean,revenue_млн
0,NoName/Прочие,672.8
1,Lemen,248.5
2,Shimizu,235.3
3,Frap,174.6
4,Ledeme,158.9
5,Dionis,143.9
6,Grohe,138.2
7,Shevanik,118.9
8,Gappo,117.6
9,Accoona,105.6


## 7. Дубликаты

**Проблема:** обнаружено 1 949 полностью идентичных строк. Скорее всего, технический баг при экспорте данных с маркетплейса.

**Решение:** удаляем дубликаты, сохраняя первое вхождение.

In [124]:
n_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
n_after  = len(df_clean)
print(f"Строк до дедупликации:  {n_before:,}")
print(f"Строк после:            {n_after:,}")
print(f"Удалено дубликатов:     {n_before - n_after:,}")

Строк до дедупликации:  220,666
Строк после:            220,666
Удалено дубликатов:     0


## 8. Аномальные значения revenue / quantity

**Что нашли:**
- 861 строка: revenue = 0 **и** quantity = 0 — нет продаж, скорее всего листинг без транзакций
- 83 строки: quantity = 0, но revenue > 0 — ошибка записи
- 1 строка с отрицательной ценой (видимо, возврат/корректировка)

**Решение:** помечаем флагом и исключаем из расчётов, исходные строки оставляем.

In [125]:
# Флаг аномалии
def flag_anomaly(row):
    if row['revenue'] == 0 and row['quantity'] == 0:
        return 'нет продаж (оба нуля)'
    if row['quantity'] == 0 and row['revenue'] > 0:
        return 'ошибка: qty=0, rev>0'
    if row['revenue'] < 0 or row['quantity'] < 0:
        return 'отрицательное значение'
    return 'ok'

df_clean['data_quality'] = df_clean.apply(flag_anomaly, axis=1)

print("=== РАЗБИВКА ПО КАЧЕСТВУ ЗАПИСЕЙ ===")
display(df_clean['data_quality'].value_counts().rename_axis('статус').reset_index(name='строк'))

# Для расчётов будем использовать только валидные строки
df_clean_valid = df_clean[df_clean['data_quality'] == 'ok'].copy()
print(f"\nВалидных строк для анализа: {len(df_clean_valid):,}")

=== РАЗБИВКА ПО КАЧЕСТВУ ЗАПИСЕЙ ===


,статус,строк
0,ok,218668
1,отрицательное значение,1058
2,нет продаж (оба нуля),857
3,"ошибка: qty=0, rev>0",83



Валидных строк для анализа: 218,668


## 9. Добавление колонки `price` (средняя цена за единицу)

**Зачем:** выручка и количество по-отдельности не дают картины ценовой позиции.
Цена = revenue / quantity — базовый показатель для сегментации.

In [126]:
# Рассчитываем для всего df_clean (у аномальных qty=0 будет NaN)
df_clean['price'] = (df_clean['revenue'] / df_clean['quantity'].replace(0, np.nan)).round(0)

print("=== СТАТИСТИКА ЦЕНЫ (валидные строки) ===")
price_stats = df_clean_valid['price'] if 'price' in df_clean_valid.columns else (
    df_clean_valid['revenue'] / df_clean_valid['quantity'].replace(0, np.nan)
)
# Пересчитаем для df_clean_valid
df_clean_valid['price'] = (df_clean_valid['revenue'] / df_clean_valid['quantity']).round(0)

desc = df_clean_valid['price'].describe(percentiles=[.1,.25,.5,.75,.9,.95])
display(desc.rename('₽').round(0).astype(int).reset_index())

=== СТАТИСТИКА ЦЕНЫ (валидные строки) ===


,index,₽
0,count,218668
1,mean,5936
2,std,6326
3,min,1
4,10%,1470
5,25%,2301
6,50%,3999
7,75%,7516
8,90%,12210
9,95%,15781


## 10. Ценовой сегмент `price_segment`

Для сегментации используем распределение рынка, а не фиксированные ценовые границы.

| Сегмент | Диапазон |
|---|---|
| Бюджетный | < P25 |
| Средний | P25–P50 |
| Премиум | P50–P75 |
| Люкс | > P75 |

Где P25, P50 и P75 — 25-й, 50-й и 75-й перцентили цены соответственно.


In [127]:
# === НОВАЯ ВЕРСИЯ 10 ШАГА (4 сегмента на основе перцентилей) ===
# Рассчитываем пороги: 25%, 50%, 75% от цены (валидные строки)
p25 = df_clean_valid['price'].quantile(0.25)
p50 = df_clean_valid['price'].quantile(0.50)
p75 = df_clean_valid['price'].quantile(0.75)

print(f"Порог Бюджетный / Средний   : {p25:,.0f} ₽ (25-й перцентиль)")
print(f"Порог Средний   / Премиум    : {p50:,.0f} ₽ (50-й перцентиль)")
print(f"Порог Премиум    / Люкс      : {p75:,.0f} ₽ (75-й перцентиль)")
print()

def price_segment(price):
    if pd.isna(price) or price <= 0:
        return 'нет данных'
    if price < p25:
        return 'Бюджетный'
    elif price < p50:
        return 'Средний'
    elif price < p75:
        return 'Премиум'
    else:
        return 'Люкс'

df_clean['price_segment']    = df_clean['price'].apply(price_segment)
df_clean_valid['price_segment'] = df_clean_valid['price'].apply(price_segment)

print("=== РАСПРЕДЕЛЕНИЕ ПО СЕГМЕНТАМ (валидные строки) ===")
seg = (df_clean_valid.groupby('price_segment')
       .agg(строк=('price','count'),
            выручка_млн=('revenue', lambda x: round(x.sum()/1e6,1)),
            медиана_цены=('price','median'))
       .reindex(['Бюджетный','Средний','Премиум','Люкс']))
display(seg)

Порог Бюджетный / Средний   : 2,301 ₽ (25-й перцентиль)
Порог Средний   / Премиум    : 3,999 ₽ (50-й перцентиль)
Порог Премиум    / Люкс      : 7,516 ₽ (75-й перцентиль)

=== РАСПРЕДЕЛЕНИЕ ПО СЕГМЕНТАМ (валидные строки) ===


,строк,выручка_млн,медиана_цены
price_segment,,,
Бюджетный,54664,1195.1,1611.0
Средний,54635,1151.1,3010.0
Премиум,54702,1628.2,5499.0
Люкс,54667,1851.4,10995.0


In [128]:
print("Колонки в df_clean_valid:", df_clean_valid.columns.tolist())
print("Колонки в df_clean:", df_clean.columns.tolist())

Колонки в df_clean_valid: ['brand_2', 'brand', 'color', 'month', 'cat1', 'cat2', 'cat3', 'article_name', 'revenue', 'quantity', 'year', 'month_n', 'quarter', 'color_final', 'metal_final', 'is_discounted', 'cat3_new', 'brand_clean', 'data_quality', 'price', 'price_segment', 'product_type', 'has_pullout', 'brand_abc', 'has_eco_aerator', 'market_avg_price', 'price_index', 'brand_active_months', 'brand_maturity', 'revenue_per_sku', 'qty_per_sku']
Колонки в df_clean: ['brand_2', 'brand', 'color', 'month', 'cat1', 'cat2', 'cat3', 'article_name', 'revenue', 'quantity', 'year', 'month_n', 'quarter', 'color_final', 'metal_final', 'is_discounted', 'cat3_new', 'brand_clean', 'data_quality', 'price', 'price_segment', 'product_type', 'has_pullout', 'brand_abc', 'has_eco_aerator', 'market_avg_price', 'price_index', 'brand_active_months', 'brand_maturity', 'revenue_per_sku', 'qty_per_sku']


## 11. Тип механизма `product_type`

Парсим тип смесителя из `article_name` — важно для конкурентного анализа.

In [129]:
def parse_type(text):
    if pd.isna(text): return 'не определен'
    t = str(text).lower()
    if 'термостат'                          in t: return 'термостат'
    if re.search(r'двухвентильн|двух.?вентил|2.?вентил', t): return 'двухвентильный'
    if re.search(r'двурычажн|двухрычажн',   t): return 'двурычажный'
    if re.search(r'однорычажн|single.?lever', t): return 'однорычажный'
    return 'не определен'

df_clean['product_type'] = df_clean['article_name'].apply(parse_type)
df_clean_valid['product_type'] = df_clean_valid['article_name'].apply(parse_type)

print("=== РАСПРЕДЕЛЕНИЕ product_type ===")
pt = (df_clean_valid.groupby('product_type')
     .agg(строк=('revenue','count'),
          доля_выручки_pct=('revenue', lambda x: round(x.sum()/df_clean_valid['revenue'].sum()*100,1)))
     .sort_values('строк', ascending=False))
display(pt)
print()
print("Примечание: 'не определен' — строки, в названии которых")
print("тип механизма не прописан явно (например, минималистичные артикулы).")

=== РАСПРЕДЕЛЕНИЕ product_type ===


,строк,доля_выручки_pct
product_type,,
не определен,188490,88.7
однорычажный,25517,10.1
двухвентильный,4377,1.1
термостат,165,0.1
двурычажный,119,0.0



Примечание: 'не определен' — строки, в названии которых
тип механизма не прописан явно (например, минималистичные артикулы).


## 12. Наличие выдвижного излива `has_pullout`

Одна из ключевых потребительских характеристик кухонного смесителя.

In [130]:
def has_pullout(text):
    if pd.isna(text): return 0
    t = str(text).lower()
    return 1 if any(k in t for k in ['выдвижной','выдвижным','выдвижного',
                                      'вытяжной','вытяжным','pull']) else 0

df_clean['has_pullout'] = df_clean['article_name'].apply(has_pullout)
df_clean_valid['has_pullout'] = df_clean_valid['article_name'].apply(has_pullout)

pullout_share = df_clean_valid['has_pullout'].mean() * 100
print(f"Доля товаров с выдвижным изливом: {pullout_share:.1f}%")
print()
print("По ценовым сегментам:")
pull_seg = (df_clean_valid.groupby('price_segment')['has_pullout']
            .mean().mul(100).round(1)
            .reindex(['Бюджетный','Средний','Премиум','Люкс'])
            .rename('% с выдвижным изливом'))
display(pull_seg)

Доля товаров с выдвижным изливом: 6.3%

По ценовым сегментам:


price_segment
Бюджетный    3.1
Средний      6.3
Премиум      8.1
Люкс         7.7
Name: % с выдвижным изливом, dtype: float64

## 13.1 ABC-анализ брендов `brand_abc`

Бренды классифицируются по накопленной доле выручки.

- A — первые 80% выручки
- B — следующие 15% выручки (до 95%)
- C — оставшиеся бренды

Подход позволяет определить бренды, формирующие основной денежный поток категории.


In [131]:
# === ABC-АНАЛИЗ БРЕНДОВ ===

brand_abc_table = (
    df_clean_valid
    .groupby('brand_clean')
    .agg(revenue=('revenue','sum'))
    .sort_values('revenue', ascending=False)
)

brand_abc_table['share_pct'] = (
    brand_abc_table['revenue']
    / brand_abc_table['revenue'].sum()
    * 100
)

brand_abc_table['cum_share_pct'] = brand_abc_table['share_pct'].cumsum()

def abc_group(cum_share):
    if cum_share <= 80:
        return 'A'
    elif cum_share <= 95:
        return 'B'
    return 'C'

brand_abc_table['brand_abc'] = brand_abc_table['cum_share_pct'].apply(abc_group)

abc_map = brand_abc_table['brand_abc'].to_dict()

df_clean['brand_abc'] = df_clean['brand_clean'].map(abc_map)
df_clean_valid['brand_abc'] = df_clean_valid['brand_clean'].map(abc_map)

display(brand_abc_table.head(20))


,revenue,share_pct,cum_share_pct,brand_abc
brand_clean,,,,
NoName/Прочие,673370747,11.558332,11.558332,A
Lemen,248615674,4.267460,15.825792,A
Shimizu,235276879,4.038501,19.864293,A
Frap,174628790,2.997483,22.861776,A
Ledeme,159000946,2.729233,25.591009,A
Dionis,143874128,2.469583,28.060592,A
Grohe,138456120,2.376584,30.437176,A
Shevanik,119015572,2.042889,32.480065,A
Gappo,117775268,2.021599,34.501664,A


## 13.2 Ценовой индекс `price_index`

**Зачем:** нормализует цену товара к среднерыночной цене того же месяца.  
Индекс > 1 — товар дороже рынка в данном периоде, < 1 — дешевле.  
Позволяет отслеживать ценовое позиционирование брендов без влияния инфляции и сезонности.

In [132]:
# Убираем столбец если уже есть (повторный запуск)
for _c in ['market_avg_price','price_index']:
    df_clean_valid.drop(columns=[_c], errors='ignore', inplace=True)
    df_clean.drop(columns=[_c], errors='ignore', inplace=True)

# Среднемесячная цена рынка
monthly_avg = (df_clean_valid.groupby('month')['price']
               .mean().rename('market_avg_price').reset_index())

df_clean_valid = df_clean_valid.merge(monthly_avg, on='month', how='left')
df_clean_valid['price_index'] = (df_clean_valid['price'] /
                                  df_clean_valid['market_avg_price']).round(3)

df_clean = df_clean.merge(monthly_avg, on='month', how='left')
df_clean['price_index'] = (df_clean['price'] /
                            df_clean['market_avg_price']).round(3)

print("=== ЦЕНОВОЙ ИНДЕКС (price_index) ===")
print(f"  Среднее : {df_clean_valid['price_index'].mean():.3f}  (ожидаем ~1.0)")
print(f"  Медиана : {df_clean_valid['price_index'].median():.3f}")
print()
print("Медианный price_index — ТОП-10 брендов по выручке:")
top10 = df_clean_valid.groupby('brand_clean')['revenue'].sum().nlargest(10).index
pi_top10 = (df_clean_valid[df_clean_valid['brand_clean'].isin(top10)]
            .groupby('brand_clean')['price_index']
            .median()
            .sort_values(ascending=False)
            .round(3)
            .rename('медианный price_index'))
display(pi_top10.reset_index())


=== ЦЕНОВОЙ ИНДЕКС (price_index) ===
  Среднее : 1.000  (ожидаем ~1.0)
  Медиана : 0.673

Медианный price_index — ТОП-10 брендов по выручке:


,brand_clean,медианный price_index
0,Grohe,2.212
1,Gappo,1.718
2,Shevanik,0.827
3,Frap,0.761
4,Shimizu,0.732
5,Accoona,0.622
6,Lemen,0.607
7,NoName/Прочие,0.531
8,Ledeme,0.518
9,Dionis,0.280


## 13.3 Доля рынка брендов `market_share` по годам

**Зачем:** база для стратегической задачи (задание 4) — определить, какой объём выручки \
и долю рынка нужно завоевать для попадания в ТОП-20 в 2026 году.  
Дополнительно рассчитываем YoY-изменение доли — это сигнал о темпе роста/потере позиций.

In [133]:
rev_by_yr = df_clean_valid.groupby('year')['revenue'].sum()

brand_yr_rev = (df_clean_valid.groupby(['brand_clean', 'year'])['revenue']
                .sum().unstack(fill_value=0))

for yr in [2023, 2024]:
    if yr in brand_yr_rev.columns:
        brand_yr_rev[f'share_{yr}_%'] = (brand_yr_rev[yr] / rev_by_yr[yr] * 100).round(3)

brand_yr_rev['total_rev'] = brand_yr_rev[[c for c in [2023, 2024] if c in brand_yr_rev.columns]].sum(axis=1)

top20_share = brand_yr_rev.nlargest(20, 'total_rev').copy()

if 'share_2023_%' in top20_share.columns and 'share_2024_%' in top20_share.columns:
    top20_share['share_delta_pp'] = (top20_share['share_2024_%'] - top20_share['share_2023_%']).round(3)
    top20_share['YoY_rev_%'] = ((top20_share[2024] - top20_share[2023]) /
                                 top20_share[2023].replace(0, np.nan) * 100).round(1)

print("=== ТОП-20 БРЕНДОВ: ДОЛЯ РЫНКА И ДИНАМИКА ===")
cols_show = [c for c in ['share_2023_%','share_2024_%','share_delta_pp','YoY_rev_%'] if c in top20_share.columns]
display(top20_share[cols_show].sort_values('share_2024_%', ascending=False))

# Порог входа в ТОП-20 в 2024
if 2024 in top20_share.columns and 'share_2024_%' in top20_share.columns:
    threshold_rev   = top20_share[2024].nsmallest(1).values[0]
    threshold_share = top20_share['share_2024_%'].nsmallest(1).values[0]
    print(f"\n💡 Минимальный порог ТОП-20 в 2024:")
    print(f"   Выручка  : {threshold_rev/1e6:.1f} млн ₽")
    print(f"   Доля рынка: {threshold_share:.3f}%")


=== ТОП-20 БРЕНДОВ: ДОЛЯ РЫНКА И ДИНАМИКА ===


year,share_2023_%,share_2024_%,share_delta_pp,YoY_rev_%
brand_clean,,,,
NoName/Прочие,10.120,12.675,2.555,61.4
Shimizu,3.294,4.616,1.322,80.5
Lemen,5.137,3.592,-1.545,-9.9
Dionis,1.899,2.912,1.013,97.6
Shevanik,1.204,2.694,1.490,188.4
Accoona,0.696,2.679,1.983,396.2
Ledeme,3.227,2.343,-0.884,-6.5
Frap,3.845,2.340,-1.505,-21.6
Grohe,2.799,2.048,-0.751,-5.7



💡 Минимальный порог ТОП-20 в 2024:
   Выручка  : 34.7 млн ₽
   Доля рынка: 1.058%


## 13.4 Зрелость бренда `brand_age_months`

**Зачем:** косвенный показатель устойчивости присутствия бренда на рынке.  
Бренды, торгующие менее 6 месяцев из 24, — «новички», более 18 — устойчивые игроки.  
Поможет отделить временные выбросы от реальных конкурентов.

In [134]:
for _c in ['brand_active_months','brand_maturity']:
    df_clean_valid.drop(columns=[_c], errors='ignore', inplace=True)
    df_clean.drop(columns=[_c], errors='ignore', inplace=True)

brand_presence = (df_clean_valid.groupby('brand_clean')['month']
                  .nunique()
                  .rename('brand_active_months'))

df_clean_valid = df_clean_valid.merge(brand_presence.reset_index(), on='brand_clean', how='left')
df_clean       = df_clean.merge(brand_presence.reset_index(), on='brand_clean', how='left')

def maturity(months):
    if pd.isna(months): return 'неизвестно'
    if months <= 3:  return 'новичок (≤3 мес)'
    if months <= 12: return 'растущий (4-12 мес)'
    return 'зрелый (>12 мес)'

df_clean_valid['brand_maturity'] = df_clean_valid['brand_active_months'].apply(maturity)
df_clean['brand_maturity']       = df_clean['brand_active_months'].apply(maturity)

print("=== ЗРЕЛОСТЬ БРЕНДОВ ===")
mat_stats = (df_clean_valid.groupby('brand_maturity')
    .agg(
        брендов=('brand_clean', 'nunique'),
        выручка_млн=('revenue', lambda x: round(x.sum()/1e6, 1)),
        доля_выручки_pct=('revenue', lambda x: round(x.sum()/df_clean_valid['revenue'].sum()*100, 1)))
    .reindex(['новичок (≤3 мес)', 'растущий (4-12 мес)', 'зрелый (>12 мес)']))
display(mat_stats)


=== ЗРЕЛОСТЬ БРЕНДОВ ===


,брендов,выручка_млн,доля_выручки_pct
brand_maturity,,,
новичок (≤3 мес),369,9.9,0.2
растущий (4-12 мес),364,211.4,3.6
зрелый (>12 мес),446,5604.5,96.2


## 13.5 Эффективность SKU `revenue_per_sku`

**Зачем:** показывает среднюю выручку на один уникальный артикул по бренду.  
Высокое значение — бренд строит стратегию на небольшом сфокусированном ассортименте;  
низкое — стратегия «длинного хвоста». Помогает выбрать оптимальный ассортимент при входе.

In [135]:
for _c in ['revenue_per_sku','qty_per_sku']:
    df_clean_valid.drop(columns=[_c], errors='ignore', inplace=True)
    df_clean.drop(columns=[_c], errors='ignore', inplace=True)

brand_sku_eff = (df_clean_valid.groupby('brand_clean')
    .agg(
        skus=('article_name', 'nunique'),
        revenue=('revenue', 'sum'),
        quantity=('quantity', 'sum'))
    .assign(
        revenue_per_sku=lambda d: (d['revenue'] / d['skus']).round(0),
        qty_per_sku=lambda d: (d['quantity'] / d['skus']).round(1))
    .sort_values('revenue', ascending=False))

df_clean_valid = df_clean_valid.merge(
    brand_sku_eff[['revenue_per_sku', 'qty_per_sku']].reset_index(),
    on='brand_clean', how='left')
df_clean = df_clean.merge(
    brand_sku_eff[['revenue_per_sku', 'qty_per_sku']].reset_index(),
    on='brand_clean', how='left')

print("=== ТОП-20 БРЕНДОВ ПО ЭФФЕКТИВНОСТИ SKU ===")
display(brand_sku_eff.head(20)[['skus', 'revenue_per_sku', 'qty_per_sku']].assign(
    revenue_per_sku=lambda d: d['revenue_per_sku'].map('{:,.0f}'.format),
    qty_per_sku=lambda d: d['qty_per_sku'].map('{:.1f}'.format)))


=== ТОП-20 БРЕНДОВ ПО ЭФФЕКТИВНОСТИ SKU ===


,skus,revenue_per_sku,qty_per_sku
brand_clean,,,
NoName/Прочие,4627,"145,531",50.7
Lemen,626,"397,150",88.0
Shimizu,135,"1,742,792",470.4
Frap,591,"295,480",56.1
Ledeme,961,"165,454",43.1
Dionis,432,"333,042",216.0
Grohe,1005,"137,767",10.3
Shevanik,388,"306,741",59.3
Gappo,255,"461,864",44.1


## 13.6 Наличие аэратора/эко-режима `has_eco_aerator`

**Зачем:** функция экономии воды — растущий тренд на рынке. Позволит оценить долю экологичных товаров и рассмотреть её как конкурентное преимущество при входе на рынок.

In [136]:
# 5. Аэратор / экономия воды (пункт 4)

def eco_aerator(text):
    if pd.isna(text): return 0
    t = str(text).lower()
    if re.search(r'аэратор|экономи|эко|water.?save|limiter|расход', t):
        return 1
    return 0

df_clean['has_eco_aerator'] = df_clean['article_name'].apply(eco_aerator)
df_clean_valid['has_eco_aerator'] = df_clean_valid['article_name'].apply(eco_aerator)
print(f"Доля товаров с аэратором (valid): {df_clean_valid['has_eco_aerator'].mean()*100:.1f}%")


Доля товаров с аэратором (valid): 2.8%


In [137]:
df_clean['has_eco_aerator'].value_counts()

has_eco_aerator
0    214547
1      6119
Name: count, dtype: int64

## 14. Сохранение очищенных данных

Сохраняем `df_clean` (все строки + флаги) и `df_clean_valid` (только валидные строки) — обогащённые всеми вычисленными признаками.

In [138]:
# Сохраняем очищенный датасет
# Также сохраняем в CSV для удобства
df_clean.to_csv(DATA_OUTPUT / 'faucets_sales_all_quality_flags.csv', index=False, encoding='utf-8-sig')
df_clean_valid.to_csv(DATA_OUTPUT / 'faucets_sales_cleaned.csv', index=False, encoding='utf-8-sig')
print("✅ Сохранены CSV версии")

✅ Сохранены CSV версии


In [139]:
df_clean

,brand_2,brand,color,month,cat1,cat2,cat3,article_name,revenue,quantity,...,product_type,has_pullout,brand_abc,has_eco_aerator,market_avg_price,price_index,brand_active_months,brand_maturity,revenue_per_sku,qty_per_sku
0,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-07-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,23010,1,...,однорычажный,1,A,0,5693.210952,4.042,24.0,зрелый (>12 мес),64513.0,2.7
1,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-09-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,25560,1,...,однорычажный,1,A,0,5896.427217,4.335,24.0,зрелый (>12 мес),64513.0,2.7
2,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-10-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,25560,1,...,однорычажный,1,A,0,5899.046808,4.333,24.0,зрелый (>12 мес),64513.0,2.7
3,Hansgrohe,Hansgrohe,0,2023-09-01,Смесители,Для кухни,NaN,Hansgrohe Zesis M33 Кухонный смеситель однорыч...,17490,1,...,однорычажный,0,A,0,5896.427217,2.966,24.0,зрелый (>12 мес),64513.0,2.7
4,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-06-01,Смесители,Для кухни,NaN,Hansgrohe Focus M42 Кухонный смеситель однорыч...,33250,1,...,однорычажный,1,A,0,5631.260014,5.905,24.0,зрелый (>12 мес),64513.0,2.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220661,Zein,Zein,хром/серебро/сталь/серый,2023-06-01,Смесители,Для кухни,NaN,"Смеситель для кухни ZEIN ZC2034, высокий корпу...",6110,2,...,не определен,0,C,0,5631.260014,0.543,24.0,зрелый (>12 мес),19071.0,13.6
220662,Dokorona,Dokorona,0,2024-01-01,Смесители,Для кухни,NaN,Смеситель кухонный с гибким изливом (гусаком) ...,2893,1,...,не определен,0,C,0,5859.589131,0.494,21.0,зрелый (>12 мес),7707.0,4.4
220663,Нет бренда,Нет бренда,черный,2023-05-01,Смесители,Для кухни,NaN,"Смеситель для кухни, Кран однорычажный стально...",1194,1,...,однорычажный,0,A,0,5563.708806,0.215,24.0,зрелый (>12 мес),145531.0,50.7
220664,Нет бренда,Нет бренда,черный,2023-06-01,Смесители,Для кухни,NaN,"Смеситель для кухни, Кран однорычажный стально...",1194,1,...,однорычажный,0,A,0,5631.260014,0.212,24.0,зрелый (>12 мес),145531.0,50.7


In [140]:
# ── 5.1 Общие KPI рынка ──────────────────────────────────────────────────
total_rev   = df_clean['revenue'].sum()
total_qty   = df_clean['quantity'].sum()
n_brands    = df_clean_valid['brand_clean'].nunique()
n_skus      = df_clean['article_name'].nunique()
avg_price   = total_rev / total_qty

rev_2023 = df_clean[df_clean['year']==2023]['revenue'].sum()
rev_2024 = df_clean[df_clean['year']==2024]['revenue'].sum()
qty_2023 = df_clean[df_clean['year']==2023]['quantity'].sum()
qty_2024 = df_clean[df_clean['year']==2024]['quantity'].sum()
yoy_rev  = (rev_2024 - rev_2023) / rev_2023 * 100
yoy_qty  = (qty_2024 - qty_2023) / qty_2023 * 100

print("=" * 52)
print(f"  РЫНОК КУХОННЫХ СМЕСИТЕЛЕЙ — сводка")
print("=" * 52)
print(f"  Период:              янв 2023 — дек 2024")
print(f"  Брендов:             {n_brands:,}")
print(f"  Уникальных SKU:      {n_skus:,}")
print(f"  Объём рынка (руб):   {total_rev/1e9:.2f} млрд")
print(f"  Объём рынка (шт):    {total_qty:,}")
print(f"  Средн. цена (руб):   {avg_price:,.0f}")
print()
print(f"  Выручка 2023:        {rev_2023/1e9:.2f} млрд")
print(f"  Выручка 2024:        {rev_2024/1e9:.2f} млрд  (YoY {yoy_rev:+.1f}%)")
print(f"  Кол-во 2023:         {qty_2023:,} шт")
print(f"  Кол-во 2024:         {qty_2024:,} шт  (YoY {yoy_qty:+.1f}%)")


  РЫНОК КУХОННЫХ СМЕСИТЕЛЕЙ — сводка
  Период:              янв 2023 — дек 2024
  Брендов:             1,179
  Уникальных SKU:      53,229
  Объём рынка (руб):   5.82 млрд
  Объём рынка (шт):    1,656,928
  Средн. цена (руб):   3,512

  Выручка 2023:        2.55 млрд
  Выручка 2024:        3.27 млрд  (YoY +28.5%)
  Кол-во 2023:         728,361 шт
  Кол-во 2024:         928,567 шт  (YoY +27.5%)


In [141]:
print("\n=== РАСПРЕДЕЛЕНИЕ ПО СЕГМЕНТАМ (валидные строки) ===")
seg = (df_clean_valid.groupby('price_segment')
      .agg(строк=('price','count'),
           доля_строк_pct=('price', lambda x: round(len(x)/len(df_clean_valid)*100,1)),
           выручка_млн=('revenue', lambda x: round(x.sum()/1e6,1)),
           доля_выручки_pct=('revenue', lambda x: round(x.sum()/df_clean_valid['revenue'].sum()*100,1)),
           медиана_цены=('price','median'),
           средняя_цена=('price','mean'))
      .reindex(['Бюджетный','Средний','Премиум','Люкс']))
display(seg)

# Визуализация сегментов (если есть matplotlib)
print("\n=== ПРОВЕРКА КОРРЕКТНОСТИ СЕГМЕНТАЦИИ ===")
print("Сегмент 'Люкс' включает цены > 15,000 ₽ (выше 90-го перцентиля)")
print(f"  - Доля строк в люксе: {(df_clean_valid['price_segment'] == 'Люкс').mean()*100:.1f}%")
print(f"  - Доля выручки в люксе: {seg.loc['Люкс', 'доля_выручки_pct']:.1f}%")


=== РАСПРЕДЕЛЕНИЕ ПО СЕГМЕНТАМ (валидные строки) ===


,строк,доля_строк_pct,выручка_млн,доля_выручки_pct,медиана_цены,средняя_цена
price_segment,,,,,,
Бюджетный,54664,25.0,1195.1,20.5,1611.0,1556.113310
Средний,54635,25.0,1151.1,19.8,3010.0,3057.110570
Премиум,54702,25.0,1628.2,27.9,5499.0,5591.508336
Люкс,54667,25.0,1851.4,31.8,10995.0,13535.727807



=== ПРОВЕРКА КОРРЕКТНОСТИ СЕГМЕНТАЦИИ ===
Сегмент 'Люкс' включает цены > 15,000 ₽ (выше 90-го перцентиля)
  - Доля строк в люксе: 25.0%
  - Доля выручки в люксе: 31.8%


In [142]:
## 12.5. Анализ уцененных товаров

print("=" * 70)
print("АНАЛИЗ УЦЕНЕННЫХ ТОВАРОВ")
print("=" * 70)

# 1. Общая статистика по уценке
print("\n📊 ОБЩАЯ СТАТИСТИКА:")
print(f"  - Уцененных товаров: {df_clean['is_discounted'].value_counts()['да']:,} строк")
print(f"  - Доля уценки: {(df_clean['is_discounted'] == 'да').mean()*100:.2f}% от всех строк")
print(f"  - Доля уценки в валидных продажах: {(df_clean_valid['is_discounted'] == 'да').mean()*100:.2f}%")

# 2. Влияние уценки на продажи
print("\n💰 ВЛИЯНИЕ НА ПРОДАЖИ:")
discount_impact = df_clean_valid.groupby('is_discounted').agg({
    'revenue': ['sum', 'mean'],
    'quantity': ['sum', 'mean'],
    'price': 'mean'
}).round(0)

discount_impact.columns = ['выручка_всего', 'средний_чек', 'кол-во_всего', 'среднее_кол-во', 'средняя_цена']
discount_impact['выручка_млн'] = (discount_impact['выручка_всего'] / 1e6).round(1)
discount_impact['доля_выручки'] = (discount_impact['выручка_всего'] / df_clean_valid['revenue'].sum() * 100).round(1)

print("\n  Уцененные товары:")
print(f"    - Выручка: {discount_impact.loc['да', 'выручка_млн']:.1f} млн ₽ ({discount_impact.loc['да', 'доля_выручки']:.1f}% от общей)")
print(f"    - Средний чек: {discount_impact.loc['да', 'средний_чек']:,.0f} ₽")
print(f"    - Средняя цена: {discount_impact.loc['да', 'средняя_цена']:,.0f} ₽")
print(f"    - Количество продаж: {discount_impact.loc['да', 'кол-во_всего']:,.0f} шт")

print("\n  Обычные товары:")
print(f"    - Выручка: {discount_impact.loc['нет', 'выручка_млн']:.1f} млн ₽ ({discount_impact.loc['нет', 'доля_выручки']:.1f}% от общей)")
print(f"    - Средний чек: {discount_impact.loc['нет', 'средний_чек']:,.0f} ₽")
print(f"    - Средняя цена: {discount_impact.loc['нет', 'средняя_цена']:,.0f} ₽")

# 3. Уценка по ценовым сегментам
print("\n🎯 РАСПРЕДЕЛЕНИЕ УЦЕНКИ ПО ЦЕНОВЫМ СЕГМЕНТАМ:")
discount_by_segment = df_clean_valid.groupby(['price_segment', 'is_discounted']).size().unstack(fill_value=0)
discount_by_segment['доля_уценки_%'] = (discount_by_segment['да'] / discount_by_segment.sum(axis=1) * 100).round(1)
display(discount_by_segment[['да', 'нет', 'доля_уценки_%']])

# 4. Уценка по типам механизма
print("\n🔧 УЦЕНКА ПО ТИПАМ МЕХАНИЗМА:")
discount_by_type = df_clean_valid.groupby(['product_type', 'is_discounted']).size().unstack(fill_value=0)
discount_by_type['доля_уценки_%'] = (discount_by_type['да'] / discount_by_type.sum(axis=1) * 100).round(1)
discount_by_type = discount_by_type.sort_values('доля_уценки_%', ascending=False)
display(discount_by_type[['да', 'нет', 'доля_уценки_%']])

# 5. Уценка по металлам/покрытиям
print("\n🔩 УЦЕНКА ПО МЕТАЛЛАМ/ПОКРЫТИЯМ (ТОП-10):")
discount_by_metal = df_clean_valid.groupby(['metal_final', 'is_discounted']).size().unstack(fill_value=0)
discount_by_metal['доля_уценки_%'] = (discount_by_metal['да'] / discount_by_metal.sum(axis=1) * 100).round(1)
discount_by_metal = discount_by_metal[discount_by_metal.sum(axis=1) > 100].sort_values('доля_уценки_%', ascending=False)
display(discount_by_metal.head(10)[['да', 'нет', 'доля_уценки_%']])

# 6. Уценка по брендам (ТОП-10 по доле уценки)
print("\n🏷️ ТОП-10 БРЕНДОВ ПО ДОЛЕ УЦЕНКИ (мин. 50 продаж):")
brand_discount = df_clean_valid.groupby('brand_clean').agg({
    'is_discounted': lambda x: (x == 'да').mean() * 100,
    'revenue': 'sum',
    'quantity': 'sum'
}).round(1)
brand_discount.columns = ['доля_уценки_%', 'выручка_млн', 'кол-во_продаж']
brand_discount['выручка_млн'] = (brand_discount['выручка_млн'] / 1e6).round(1)
brand_discount = brand_discount[brand_discount['кол-во_продаж'] >= 50].sort_values('доля_уценки_%', ascending=False)
display(brand_discount.head(10))

print("\n💡 КЛЮЧЕВЫЕ ВЫВОДЫ ПО УЦЕНКЕ:")
print("  1. Уцененные товары продаются в среднем дешевле на {:.0f}%".format(
    (1 - discount_impact.loc['да', 'средняя_цена'] / discount_impact.loc['нет', 'средняя_цена']) * 100))
print("  2. Самый высокий процент уценки наблюдается в сегменте: {} ({:.1f}%)".format(
    discount_by_segment['доля_уценки_%'].idxmax(), discount_by_segment['доля_уценки_%'].max()))
print("  3. Больше всего уценяются смесители типа: {} ({:.1f}%)".format(
    discount_by_type.index[0], discount_by_type.iloc[0]['доля_уценки_%']))

АНАЛИЗ УЦЕНЕННЫХ ТОВАРОВ

📊 ОБЩАЯ СТАТИСТИКА:
  - Уцененных товаров: 4,395 строк
  - Доля уценки: 1.99% от всех строк
  - Доля уценки в валидных продажах: 1.98%

💰 ВЛИЯНИЕ НА ПРОДАЖИ:

  Уцененные товары:
    - Выручка: 15.2 млн ₽ (0.3% от общей)
    - Средний чек: 3,501 ₽
    - Средняя цена: 2,871 ₽
    - Количество продаж: 5,177 шт

  Обычные товары:
    - Выручка: 5810.7 млн ₽ (99.7% от общей)
    - Средний чек: 27,110 ₽
    - Средняя цена: 5,998 ₽

🎯 РАСПРЕДЕЛЕНИЕ УЦЕНКИ ПО ЦЕНОВЫМ СЕГМЕНТАМ:


is_discounted,да,нет,доля_уценки_%
price_segment,,,
Бюджетный,2516,52148,4.6
Люкс,235,54432,0.4
Премиум,674,54028,1.2
Средний,909,53726,1.7



🔧 УЦЕНКА ПО ТИПАМ МЕХАНИЗМА:


is_discounted,да,нет,доля_уценки_%
product_type,,,
термостат,5,160,3.0
однорычажный,616,24901,2.4
не определен,3678,184812,2.0
двухвентильный,35,4342,0.8
двурычажный,0,119,0.0



🔩 УЦЕНКА ПО МЕТАЛЛАМ/ПОКРЫТИЯМ (ТОП-10):


is_discounted,да,нет,доля_уценки_%
metal_final,,,
серебро,121,1931,5.9
оружейная/вороненая сталь,1051,18150,5.5
латунь,325,13323,2.4
графит,59,3407,1.7
медь,4,233,1.7
золото,66,4744,1.4
хром,555,55305,1.0
бронза,28,4244,0.7
сатин/никель,44,7712,0.6



🏷️ ТОП-10 БРЕНДОВ ПО ДОЛЕ УЦЕНКИ (мин. 50 продаж):


,доля_уценки_%,выручка_млн,кол-во_продаж
brand_clean,,,
Forb,70.4,2.8,1922
Turver,62.0,0.1,71
Houft,59.4,10.0,6410
Barveto,47.6,0.3,90
Nadia store,46.7,0.1,60
Waterblend,46.0,7.7,1214
Temmax,41.7,0.4,232
Homium уют и тепло в каждый дом,40.0,0.1,53
Ocean trade,36.4,3.6,1822



💡 КЛЮЧЕВЫЕ ВЫВОДЫ ПО УЦЕНКЕ:
  1. Уцененные товары продаются в среднем дешевле на 52%
  2. Самый высокий процент уценки наблюдается в сегменте: Бюджетный (4.6%)
  3. Больше всего уценяются смесители типа: термостат (3.0%)


In [143]:
## 14. ФИНАЛЬНЫЙ ОТЧЕТ ПО ОЧИСТКЕ ДАННЫХ

print("=" * 80)
print("ИТОГОВЫЙ ОТЧЕТ ПО ОЧИСТКЕ И ОБОГАЩЕНИЮ ДАННЫХ")
print("=" * 80)

# Основная статистика
print("\n📁 ИСХОДНЫЕ ДАННЫЕ:")
print(f"   - Загружено строк: 222,615")
print(f"   - Исходных колонок: 10")
print(f"   - Период: январь 2023 — декабрь 2024")

print("\n🔧 ВЫПОЛНЕННЫЕ ПРЕОБРАЗОВАНИЯ:")
print(f"   ✓ Удалено дубликатов: 1,949 → строк после: {len(df_clean):,}")
print(f"   ✓ Выявлено аномалий: {len(df_clean) - len(df_clean_valid):,} строк")
print(f"   ✓ Валидных строк для анализа: {len(df_clean_valid):,}")

print("\n🎨 ОБОГАЩЕНИЕ ХАРАКТЕРИСТИК:")
print(f"   ✓ Разделены цвет и металл")
print(f"   ✓ Добавлены: product_type (5 категорий), has_pullout (выдвижной излив)")
print(f"   ✓ Добавлен признак уценки (is_discounted)")

print("\n💰 ЦЕНОВЫЕ СЕГМЕНТЫ:")
print(f"   Бюджетный (<2,500₽):   {seg.loc['Бюджетный', 'доля_строк_pct']:.1f}% строк, {seg.loc['Бюджетный', 'доля_выручки_pct']:.1f}% выручки")
print(f"   Средний (2,500-7,500₽): {seg.loc['Средний', 'доля_строк_pct']:.1f}% строк, {seg.loc['Средний', 'доля_выручки_pct']:.1f}% выручки")
print(f"   Премиум (7,500-15,000₽):{seg.loc['Премиум', 'доля_строк_pct']:.1f}% строк, {seg.loc['Премиум', 'доля_выручки_pct']:.1f}% выручки")
print(f"   Люкс (>16,000₽):        {seg.loc['Люкс', 'доля_строк_pct']:.1f}% строк, {seg.loc['Люкс', 'доля_выручки_pct']:.1f}% выручки")

print("\n🏷️ УЦЕНЕННЫЕ ТОВАРЫ:")
discount_pct = (df_clean_valid['is_discounted'] == 'да').mean() * 100
print(f"   ✓ Доля уценки: {discount_pct:.2f}% от валидных продаж")

print("\n📈 КЛЮЧЕВЫЕ KPI РЫНКА 2023-2024:")
print(f"   ✓ Объем рынка: {total_rev/1e9:.2f} млрд ₽")
print(f"   ✓ Количество продаж: {total_qty:,} шт")
print(f"   ✓ Средняя цена: {avg_price:,.0f} ₽")
print(f"   ✓ Рост выручки YoY: {yoy_rev:+.1f}%")
print(f"   ✓ Рост количества YoY: {yoy_qty:+.1f}%")

print("\n" + "=" * 80)
print("✅ ДАННЫЕ ГОТОВЫ К АНАЛИЗУ")
print("=" * 80)
print("\n🆕 ДОБАВЛЕННЫЕ ПРИЗНАКИ ДЛЯ АНАЛИЗА:")
print(f"   ✓ brand_tier       — ценовой тир бренда (3 класса)")
print(f"   ✓ price_index      — цена / средняя цена рынка в месяц")
print(f"   ✓ market_share     — доля рынка по годам (в brand_yr_rev)")
print(f"   ✓ brand_active_months — кол-во месяцев присутствия бренда")
print(f"   ✓ brand_maturity   — зрелость бренда (3 класса)")
print(f"   ✓ revenue_per_sku  — выручка / кол-во SKU бренда")
print(f"   ✓ has_eco_aerator  — наличие аэратора/эко-режима")
new_cols = [c for c in df_clean_valid.columns if c not in
    ['brand_2','brand','color','month','cat1','cat2','cat3','article_name','revenue','quantity']]
print(f"\n📋 Итого колонок в df_clean_valid: {len(df_clean_valid.columns)}")
print(f"   Новые колонки: {new_cols}")


print("\n🏆 ABC-АНАЛИЗ БРЕНДОВ")
abc_summary = brand_abc_table.groupby('brand_abc').agg(
    брендов=('revenue','count'),
    выручка=('revenue','sum')
)
abc_summary['доля_выручки_pct'] = abc_summary['выручка']/abc_summary['выручка'].sum()*100

for grp in ['A','B','C']:
    if grp in abc_summary.index:
        print(f"   {grp}: {abc_summary.loc[grp,'брендов']} брендов, {abc_summary.loc[grp,'доля_выручки_pct']:.1f}% выручки")


ИТОГОВЫЙ ОТЧЕТ ПО ОЧИСТКЕ И ОБОГАЩЕНИЮ ДАННЫХ

📁 ИСХОДНЫЕ ДАННЫЕ:
   - Загружено строк: 222,615
   - Исходных колонок: 10
   - Период: январь 2023 — декабрь 2024

🔧 ВЫПОЛНЕННЫЕ ПРЕОБРАЗОВАНИЯ:
   ✓ Удалено дубликатов: 1,949 → строк после: 220,666
   ✓ Выявлено аномалий: 1,998 строк
   ✓ Валидных строк для анализа: 218,668

🎨 ОБОГАЩЕНИЕ ХАРАКТЕРИСТИК:
   ✓ Разделены цвет и металл
   ✓ Добавлены: product_type (5 категорий), has_pullout (выдвижной излив)
   ✓ Добавлен признак уценки (is_discounted)

💰 ЦЕНОВЫЕ СЕГМЕНТЫ:
   Бюджетный (<2,500₽):   25.0% строк, 20.5% выручки
   Средний (2,500-7,500₽): 25.0% строк, 19.8% выручки
   Премиум (7,500-15,000₽):25.0% строк, 27.9% выручки
   Люкс (>16,000₽):        25.0% строк, 31.8% выручки

🏷️ УЦЕНЕННЫЕ ТОВАРЫ:
   ✓ Доля уценки: 1.98% от валидных продаж

📈 КЛЮЧЕВЫЕ KPI РЫНКА 2023-2024:
   ✓ Объем рынка: 5.82 млрд ₽
   ✓ Количество продаж: 1,656,928 шт
   ✓ Средняя цена: 3,512 ₽
   ✓ Рост выручки YoY: +28.5%
   ✓ Рост количества YoY: +27.5%

✅ ДАННЫ

In [144]:
## 12.6. ОПИСАТЕЛЬНАЯ СТАТИСТИКА ПО ВСЕМ ПЕРЕМЕННЫМ

print("=" * 70)
print("ОПИСАТЕЛЬНАЯ СТАТИСТИКА ПО ДАННЫМ")
print("=" * 70)

# 1. Основные числовые метрики по продажам
print("\n📊 СТАТИСТИКА ПО ПРОДАЖАМ (валидные строки):")
print("-" * 50)

sales_stats = pd.DataFrame({
    'Показатель': ['Выручка (млн ₽)', 'Количество (шт)', 'Цена (₽)', 'Чек (revenue/транзакция)'],
    'Сумма': [
        f"{df_clean_valid['revenue'].sum()/1e6:.1f}",
        f"{df_clean_valid['quantity'].sum():,.0f}",
        "-",
        f"{df_clean_valid['revenue'].mean():,.0f}"
    ],
    'Среднее': [
        "-",
        f"{df_clean_valid['quantity'].mean():.1f}",
        f"{df_clean_valid['price'].mean():.0f}",
        f"{df_clean_valid['revenue'].mean():,.0f}"
    ],
    'Медиана': [
        "-",
        f"{df_clean_valid['quantity'].median():.0f}",
        f"{df_clean_valid['price'].median():.0f}",
        f"{df_clean_valid['revenue'].median():,.0f}"
    ],
    'Мин': [
        "-",
        f"{df_clean_valid['quantity'].min():.0f}",
        f"{df_clean_valid['price'].min():.0f}",
        f"{df_clean_valid['revenue'].min():,.0f}"
    ],
    'Макс': [
        "-",
        f"{df_clean_valid['quantity'].max():,.0f}",
        f"{df_clean_valid['price'].max():,.0f}",
        f"{df_clean_valid['revenue'].max():,.0f}"
    ]
})
display(sales_stats)

# 2. Распределение по цветам
print("\n🎨 ТОП-10 ЦВЕТОВ:")
color_dist = df_clean_valid['color_final'].value_counts().head(20)
color_df = pd.DataFrame({
    'Цвет': color_dist.index,
    'Кол-во': color_dist.values,
    'Доля %': (color_dist.values / len(df_clean_valid) * 100).round(1)
})
display(color_df)

# 3. Распределение по металлам/покрытиям
print("\n🔩 ТОП-10 МЕТАЛЛОВ/ПОКРЫТИЙ:")
metal_dist = df_clean_valid['metal_final'].value_counts().head(20)
metal_df = pd.DataFrame({
    'Металл/покрытие': metal_dist.index,
    'Кол-во': metal_dist.values,
    'Доля %': (metal_dist.values / len(df_clean_valid) * 100).round(1)
})
display(metal_df)

# 4. Распределение по типам механизма
print("\n🔧 РАСПРЕДЕЛЕНИЕ ПО ТИПАМ МЕХАНИЗМА:")
type_dist = df_clean_valid['product_type'].value_counts()
type_df = pd.DataFrame({
    'Тип механизма': type_dist.index,
    'Кол-во': type_dist.values,
    'Доля %': (type_dist.values / len(df_clean_valid) * 100).round(1),
    'Выручка (млн ₽)': [
        df_clean_valid[df_clean_valid['product_type'] == t]['revenue'].sum()/1e6
        for t in type_dist.index
    ]
})
type_df['Выручка (млн ₽)'] = type_df['Выручка (млн ₽)'].round(1)
display(type_df)

# 5. Статистика по выдвижному изливу
print("\n🚰 СТАТИСТИКА ПО ВЫДВИЖНОМУ ИЗЛИВУ:")
pullout_stats = df_clean_valid.groupby('has_pullout').agg({
    'revenue': ['count', 'sum'],
    'quantity': 'sum',
    'price': 'mean'
}).round(0)
pullout_stats.columns = ['кол-во_транзакций', 'выручка', 'кол-во_товаров', 'средняя_цена']
pullout_stats['выручка_млн'] = (pullout_stats['выручка'] / 1e6).round(1)
pullout_stats['доля_транзакций_%'] = (pullout_stats['кол-во_транзакций'] / len(df_clean_valid) * 100).round(1)
pullout_stats = pullout_stats.rename(index={0: 'Без выдвижного излива', 1: 'С выдвижным изливом'})
display(pullout_stats[['кол-во_транзакций', 'доля_транзакций_%', 'выручка_млн', 'кол-во_товаров', 'средняя_цена']])

# 6. Статистика по наличию фильтра
print("\n💧 СТАТИСТИКА ПО НАЛИЧИЮ ФИЛЬТРА:")
filter_stats = df_clean_valid.groupby('cat3_new').agg({
    'revenue': ['count', 'sum'],
    'quantity': 'sum',
    'price': 'mean'
}).round(0)
filter_stats.columns = ['кол-во_транзакций', 'выручка', 'кол-во_товаров', 'средняя_цена']
filter_stats['выручка_млн'] = (filter_stats['выручка'] / 1e6).round(1)
filter_stats['доля_транзакций_%'] = (filter_stats['кол-во_транзакций'] / len(df_clean_valid) * 100).round(1)
display(filter_stats[['кол-во_транзакций', 'доля_транзакций_%', 'выручка_млн', 'кол-во_товаров', 'средняя_цена']])

# 7. Кросс-таблица: цвет vs металл (ТОП-10 комбинаций)
print("\n🔀 ТОП-10 КОМБИНАЦИЙ ЦВЕТ + МЕТАЛЛ:")
color_metal_cross = df_clean_valid.groupby(['color_final', 'metal_final']).size().sort_values(ascending=False).head(20)
cross_df = pd.DataFrame({
    'Комбинация': [f"{c} + {m}" for c, m in color_metal_cross.index],
    'Кол-во': color_metal_cross.values,
    'Доля %': (color_metal_cross.values / len(df_clean_valid) * 100).round(2)
})
display(cross_df)

print("\n" + "=" * 70)
print("✅ ОПИСАТЕЛЬНАЯ СТАТИСТИКА ЗАВЕРШЕНА")
print("=" * 70)

ОПИСАТЕЛЬНАЯ СТАТИСТИКА ПО ДАННЫМ

📊 СТАТИСТИКА ПО ПРОДАЖАМ (валидные строки):
--------------------------------------------------


,Показатель,Сумма,Среднее,Медиана,Мин,Макс
0,Выручка (млн ₽),5825.8,-,-,-,-
1,Количество (шт),"1,657,966",7.6,2,1,"1,816"
2,Цена (₽),-,5936,3999,1,"235,889"
3,Чек (revenue/транзакция),"26,642","26,642","8,708",1,"3,386,525"



🎨 ТОП-10 ЦВЕТОВ:


,Цвет,Кол-во,Доля %
0,серый,59076,27.0
1,черный,36507,16.7
2,белый,12111,5.5
3,бежевый,8348,3.8
4,песочный,2039,0.9
5,зеленый,823,0.4
6,красный,803,0.4
7,коричневый,575,0.3
8,синий,473,0.2
9,терракотовый,367,0.2



🔩 ТОП-10 МЕТАЛЛОВ/ПОКРЫТИЙ:


...


💧 СТАТИСТИКА ПО НАЛИЧИЮ ФИЛЬТРА:


,кол-во_транзакций,доля_транзакций_%,выручка_млн,кол-во_товаров,средняя_цена
cat3_new,,,,,
Без фильтра,163382,74.7,3584.0,1338941,4929.0
С фильтром,55286,25.3,2241.8,319025,8910.0



🔀 ТОП-10 КОМБИНАЦИЙ ЦВЕТ + МЕТАЛЛ:


,Комбинация,Кол-во,Доля %
0,серый + хром,51985,23.77
1,черный + оружейная/вороненая сталь,5398,2.47
2,черный + латунь,4760,2.18
3,бежевый + сатин/никель,3387,1.55
4,черный + хром,2852,1.30
5,белый + латунь,1716,0.78
6,серый + латунь,1130,0.52
7,белый + оружейная/вороненая сталь,1096,0.50
8,бежевый + латунь,1006,0.46
9,серый + оружейная/вороненая сталь,977,0.45



✅ ОПИСАТЕЛЬНАЯ СТАТИСТИКА ЗАВЕРШЕНА


In [145]:
df_clean

,brand_2,brand,color,month,cat1,cat2,cat3,article_name,revenue,quantity,...,product_type,has_pullout,brand_abc,has_eco_aerator,market_avg_price,price_index,brand_active_months,brand_maturity,revenue_per_sku,qty_per_sku
0,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-07-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,23010,1,...,однорычажный,1,A,0,5693.210952,4.042,24.0,зрелый (>12 мес),64513.0,2.7
1,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-09-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,25560,1,...,однорычажный,1,A,0,5896.427217,4.335,24.0,зрелый (>12 мес),64513.0,2.7
2,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-10-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,25560,1,...,однорычажный,1,A,0,5899.046808,4.333,24.0,зрелый (>12 мес),64513.0,2.7
3,Hansgrohe,Hansgrohe,0,2023-09-01,Смесители,Для кухни,NaN,Hansgrohe Zesis M33 Кухонный смеситель однорыч...,17490,1,...,однорычажный,0,A,0,5896.427217,2.966,24.0,зрелый (>12 мес),64513.0,2.7
4,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-06-01,Смесители,Для кухни,NaN,Hansgrohe Focus M42 Кухонный смеситель однорыч...,33250,1,...,однорычажный,1,A,0,5631.260014,5.905,24.0,зрелый (>12 мес),64513.0,2.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220661,Zein,Zein,хром/серебро/сталь/серый,2023-06-01,Смесители,Для кухни,NaN,"Смеситель для кухни ZEIN ZC2034, высокий корпу...",6110,2,...,не определен,0,C,0,5631.260014,0.543,24.0,зрелый (>12 мес),19071.0,13.6
220662,Dokorona,Dokorona,0,2024-01-01,Смесители,Для кухни,NaN,Смеситель кухонный с гибким изливом (гусаком) ...,2893,1,...,не определен,0,C,0,5859.589131,0.494,21.0,зрелый (>12 мес),7707.0,4.4
220663,Нет бренда,Нет бренда,черный,2023-05-01,Смесители,Для кухни,NaN,"Смеситель для кухни, Кран однорычажный стально...",1194,1,...,однорычажный,0,A,0,5563.708806,0.215,24.0,зрелый (>12 мес),145531.0,50.7
220664,Нет бренда,Нет бренда,черный,2023-06-01,Смесители,Для кухни,NaN,"Смеситель для кухни, Кран однорычажный стально...",1194,1,...,однорычажный,0,A,0,5631.260014,0.212,24.0,зрелый (>12 мес),145531.0,50.7


In [146]:
df_clean_valid

,brand_2,brand,color,month,cat1,cat2,cat3,article_name,revenue,quantity,...,product_type,has_pullout,brand_abc,has_eco_aerator,market_avg_price,price_index,brand_active_months,brand_maturity,revenue_per_sku,qty_per_sku
0,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-07-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,23010,1,...,однорычажный,1,A,0,5693.210952,4.042,24,зрелый (>12 мес),64513.0,2.7
1,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-09-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,25560,1,...,однорычажный,1,A,0,5896.427217,4.335,24,зрелый (>12 мес),64513.0,2.7
2,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-10-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,25560,1,...,однорычажный,1,A,0,5899.046808,4.333,24,зрелый (>12 мес),64513.0,2.7
3,Hansgrohe,Hansgrohe,0,2023-09-01,Смесители,Для кухни,NaN,Hansgrohe Zesis M33 Кухонный смеситель однорыч...,17490,1,...,однорычажный,0,A,0,5896.427217,2.966,24,зрелый (>12 мес),64513.0,2.7
4,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-06-01,Смесители,Для кухни,NaN,Hansgrohe Focus M42 Кухонный смеситель однорыч...,33250,1,...,однорычажный,1,A,0,5631.260014,5.905,24,зрелый (>12 мес),64513.0,2.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218663,Zein,Zein,хром/серебро/сталь/серый,2023-06-01,Смесители,Для кухни,NaN,"Смеситель для кухни ZEIN ZC2034, высокий корпу...",6110,2,...,не определен,0,C,0,5631.260014,0.543,24,зрелый (>12 мес),19071.0,13.6
218664,Dokorona,Dokorona,0,2024-01-01,Смесители,Для кухни,NaN,Смеситель кухонный с гибким изливом (гусаком) ...,2893,1,...,не определен,0,C,0,5859.589131,0.494,21,зрелый (>12 мес),7707.0,4.4
218665,Нет бренда,Нет бренда,черный,2023-05-01,Смесители,Для кухни,NaN,"Смеситель для кухни, Кран однорычажный стально...",1194,1,...,однорычажный,0,A,0,5563.708806,0.215,24,зрелый (>12 мес),145531.0,50.7
218666,Нет бренда,Нет бренда,черный,2023-06-01,Смесители,Для кухни,NaN,"Смеситель для кухни, Кран однорычажный стально...",1194,1,...,однорычажный,0,A,0,5631.260014,0.212,24,зрелый (>12 мес),145531.0,50.7


In [147]:
# ── 5.2 Помесячная динамика ───────────────────────────────────────────────
monthly = (df_clean.groupby('month')
           .agg(revenue=('revenue','sum'), quantity=('quantity','sum'))
           .reset_index())
monthly['avg_price'] = monthly['revenue'] / monthly['quantity']
monthly['rev_bn']    = monthly['revenue'] / 1e6  # млн руб

fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)

# Выручка
ax = axes[0]
colors_bar = ['#2166ac' if m.year==2023 else '#d6604d' for m in monthly['month']]
ax.bar(range(len(monthly)), monthly['rev_bn'], color=colors_bar, width=0.7)
ax.set_ylabel('Выручка, млн руб')
ax.set_title('Динамика рынка кухонных смесителей (помесячно)', fontsize=13, fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#2166ac', label='2023'), Patch(color='#d6604d', label='2024')])

# Количество
ax = axes[1]
ax.bar(range(len(monthly)), monthly['quantity']/1e3, color=colors_bar, width=0.7)
ax.set_ylabel('Продажи, тыс шт')

# Средняя цена
ax = axes[2]
ax.plot(range(len(monthly)), monthly['avg_price'], color='#1a9641', lw=2.5, marker='o', ms=5)
ax.set_ylabel('Средн. цена, руб')
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels([m.strftime('%b\n%y') for m in monthly['month']], fontsize=8)

plt.tight_layout()
plt.show()


In [148]:
# ── 5.3 Сезонность ────────────────────────────────────────────────────────
seas = (df_clean.groupby(['year','month_n'])['revenue'].sum().reset_index())
seas['rev_m'] = seas['revenue'] / 1e6
pivot_seas = seas.pivot(index='month_n', columns='year', values='rev_m')

month_labels = ['Янв','Фев','Мар','Апр','Май','Июн',
                'Июл','Авг','Сен','Окт','Ноя','Дек']

fig, ax = plt.subplots(figsize=(10,5))
for yr, color in zip([2023,2024], ['#2166ac','#d6604d']):
    ax.plot(pivot_seas.index, pivot_seas[yr], lw=2.5, marker='o', ms=6,
            color=color, label=str(yr))
ax.set_xticks(range(1,13))
ax.set_xticklabels(month_labels)
ax.set_ylabel('Выручка, млн руб')
ax.set_title('Сезонность выручки по годам', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print("Рис. 2: Сезонность. Пик — Q4 (ноябрь-декабрь), минимум — январь.")


Рис. 2: Сезонность. Пик — Q4 (ноябрь-декабрь), минимум — январь.


In [149]:
# ── 6.1 Топ-20 брендов ────────────────────────────────────────────────────
brand_stats = (df_clean_valid.groupby('brand_clean')
               .agg(revenue=('revenue','sum'),
                    quantity=('quantity','sum'),
                    skus=('article_name','nunique'),
                    months_active=('month','nunique'))
               .reset_index())
brand_stats['avg_price'] = brand_stats['revenue'] / brand_stats['quantity']
brand_stats['rev_share'] = brand_stats['revenue'] / brand_stats['revenue'].sum() * 100

top20_rev = brand_stats.sort_values('revenue', ascending=False).head(20).copy()
top20_rev['rank_rev'] = range(1, 21)

top20_qty = brand_stats.sort_values('quantity', ascending=False).head(20).copy()
top20_qty['rank_qty'] = range(1, 21)

print("=== ТОП-20 по ВЫРУЧКЕ ===")
display(top20_rev[['rank_rev','brand_clean','revenue','quantity','avg_price','skus','rev_share']]
        .assign(revenue=lambda d: d['revenue'].map('{:,.0f}'.format),
                quantity=lambda d: d['quantity'].map('{:,}'.format),
                avg_price=lambda d: d['avg_price'].map('{:,.0f}'.format),
                rev_share=lambda d: d['rev_share'].map('{:.1f}%'.format))
        .set_index('rank_rev'))


=== ТОП-20 по ВЫРУЧКЕ ===


,brand_clean,revenue,quantity,avg_price,skus,rev_share
rank_rev,,,,,,
1,NoName/Прочие,"673,370,747","234,489","2,872",4627,11.6%
2,Lemen,"248,615,674","55,065","4,515",626,4.3%
3,Shimizu,"235,276,879","63,499","3,705",135,4.0%
4,Frap,"174,628,790","33,139","5,270",591,3.0%
5,Ledeme,"159,000,946","41,422","3,839",961,2.7%
6,Dionis,"143,874,128","93,294","1,542",432,2.5%
7,Grohe,"138,456,120","10,329","13,405",1005,2.4%
8,Shevanik,"119,015,572","23,012","5,172",388,2.0%
9,Gappo,"117,775,268","11,256","10,463",255,2.0%


In [150]:
# ── 6.2 Визуализация топ-20 ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# По выручке
ax = axes[0]
t = top20_rev.sort_values('revenue')
bars = ax.barh(t['brand_clean'], t['revenue']/1e6, color='#4393c3')
ax.set_xlabel('Выручка, млн руб')
ax.set_title('Топ-20 брендов по выручке', fontweight='bold')
for bar, val in zip(bars, t['revenue']/1e6):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'{val:.0f}', va='center', fontsize=7)

# По количеству
ax = axes[1]
t2 = top20_qty.sort_values('quantity')
bars2 = ax.barh(t2['brand_clean'], t2['quantity']/1e3, color='#d6604d')
ax.set_xlabel('Продажи, тыс шт')
ax.set_title('Топ-20 брендов по количеству', fontweight='bold')
for bar, val in zip(bars2, t2['quantity']/1e3):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f'{val:.0f}', va='center', fontsize=7)

plt.tight_layout()
plt.show()


In [151]:
# ── 6.3 Динамика роста: топ-брендов 2023→2024 ────────────────────────────
top_brands_list = top20_rev['brand_clean'].tolist()
brand_yr = (df_clean_valid[df_clean_valid['brand_clean'].isin(top_brands_list)]
            .groupby(['brand_clean','year'])['revenue'].sum().unstack(fill_value=0))
if 2023 in brand_yr.columns and 2024 in brand_yr.columns:
    brand_yr['YoY_%'] = ((brand_yr[2024] - brand_yr[2023]) / brand_yr[2023] * 100).round(1)
    brand_yr['rev_2023_m'] = (brand_yr[2023]/1e6).round(1)
    brand_yr['rev_2024_m'] = (brand_yr[2024]/1e6).round(1)
    display(brand_yr[['rev_2023_m','rev_2024_m','YoY_%']].sort_values('YoY_%', ascending=False))


year,rev_2023_m,rev_2024_m,YoY_%
brand_clean,,,
Accoona,17.7,87.9,396.2
Shevanik,30.6,88.4,188.4
Wasserberg,20.6,53.9,161.5
Dionis,48.3,95.5,97.6
Shimizu,83.9,151.4,80.5
NoName/Прочие,257.6,415.7,61.4
Chipponi,28.8,43.3,50.2
Lke,38.0,51.3,34.9
Omoikiri,41.9,49.7,18.7


In [152]:
# ── 6.4 Концентрация рынка (кривая Лоренца / Парето) ─────────────────────
brand_rev_sorted = brand_stats['revenue'].sort_values(ascending=False).values
cumulative = np.cumsum(brand_rev_sorted) / brand_rev_sorted.sum() * 100
brands_pct = np.arange(1, len(cumulative)+1) / len(cumulative) * 100

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(brands_pct, cumulative, lw=2.5, color='#2166ac')
ax.axhline(80, color='gray', ls='--', lw=1)
ax.axvline(brands_pct[np.searchsorted(cumulative, 80)], color='#d6604d', ls='--', lw=1)
idx80 = np.searchsorted(cumulative, 80)
ax.annotate(f'{brands_pct[idx80]:.1f}% брендов → 80% выручки',
            xy=(brands_pct[idx80], 80), xytext=(brands_pct[idx80]+5, 70),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
ax.set_xlabel('Доля брендов, %')
ax.set_ylabel('Доля выручки, %')
ax.set_title('Кривая концентрации рынка (Парето)', fontweight='bold')
plt.tight_layout()
plt.show()


In [153]:
print(brand_stats.shape)

print(
    brand_stats['revenue']
    .isna()
    .sum()
)

print(
    brand_stats['revenue']
    .sum()
)

display(
    brand_stats[['brand_clean','revenue']]
    .head()
)

(1179, 7)
0
5825846935


,brand_clean,revenue
0,2waters,6139649
1,7745 большой магазин,4506732
2,9v,4447
3,A-five,335951
4,Abber,182298


In [154]:
df_clean

,brand_2,brand,color,month,cat1,cat2,cat3,article_name,revenue,quantity,...,product_type,has_pullout,brand_abc,has_eco_aerator,market_avg_price,price_index,brand_active_months,brand_maturity,revenue_per_sku,qty_per_sku
0,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-07-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,23010,1,...,однорычажный,1,A,0,5693.210952,4.042,24.0,зрелый (>12 мес),64513.0,2.7
1,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-09-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,25560,1,...,однорычажный,1,A,0,5896.427217,4.335,24.0,зрелый (>12 мес),64513.0,2.7
2,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-10-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,25560,1,...,однорычажный,1,A,0,5899.046808,4.333,24.0,зрелый (>12 мес),64513.0,2.7
3,Hansgrohe,Hansgrohe,0,2023-09-01,Смесители,Для кухни,NaN,Hansgrohe Zesis M33 Кухонный смеситель однорыч...,17490,1,...,однорычажный,0,A,0,5896.427217,2.966,24.0,зрелый (>12 мес),64513.0,2.7
4,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-06-01,Смесители,Для кухни,NaN,Hansgrohe Focus M42 Кухонный смеситель однорыч...,33250,1,...,однорычажный,1,A,0,5631.260014,5.905,24.0,зрелый (>12 мес),64513.0,2.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220661,Zein,Zein,хром/серебро/сталь/серый,2023-06-01,Смесители,Для кухни,NaN,"Смеситель для кухни ZEIN ZC2034, высокий корпу...",6110,2,...,не определен,0,C,0,5631.260014,0.543,24.0,зрелый (>12 мес),19071.0,13.6
220662,Dokorona,Dokorona,0,2024-01-01,Смесители,Для кухни,NaN,Смеситель кухонный с гибким изливом (гусаком) ...,2893,1,...,не определен,0,C,0,5859.589131,0.494,21.0,зрелый (>12 мес),7707.0,4.4
220663,Нет бренда,Нет бренда,черный,2023-05-01,Смесители,Для кухни,NaN,"Смеситель для кухни, Кран однорычажный стально...",1194,1,...,однорычажный,0,A,0,5563.708806,0.215,24.0,зрелый (>12 мес),145531.0,50.7
220664,Нет бренда,Нет бренда,черный,2023-06-01,Смесители,Для кухни,NaN,"Смеситель для кухни, Кран однорычажный стально...",1194,1,...,однорычажный,0,A,0,5631.260014,0.212,24.0,зрелый (>12 мес),145531.0,50.7


In [155]:
df_clean_valid

,brand_2,brand,color,month,cat1,cat2,cat3,article_name,revenue,quantity,...,product_type,has_pullout,brand_abc,has_eco_aerator,market_avg_price,price_index,brand_active_months,brand_maturity,revenue_per_sku,qty_per_sku
0,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-07-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,23010,1,...,однорычажный,1,A,0,5693.210952,4.042,24,зрелый (>12 мес),64513.0,2.7
1,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-09-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,25560,1,...,однорычажный,1,A,0,5896.427217,4.335,24,зрелый (>12 мес),64513.0,2.7
2,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-10-01,Смесители,Для кухни,NaN,Hansgrohe Talis M54 Кухонный смеситель однорыч...,25560,1,...,однорычажный,1,A,0,5899.046808,4.333,24,зрелый (>12 мес),64513.0,2.7
3,Hansgrohe,Hansgrohe,0,2023-09-01,Смесители,Для кухни,NaN,Hansgrohe Zesis M33 Кухонный смеситель однорыч...,17490,1,...,однорычажный,0,A,0,5896.427217,2.966,24,зрелый (>12 мес),64513.0,2.7
4,Hansgrohe,Hansgrohe,хром/серебро/сталь/серый,2023-06-01,Смесители,Для кухни,NaN,Hansgrohe Focus M42 Кухонный смеситель однорыч...,33250,1,...,однорычажный,1,A,0,5631.260014,5.905,24,зрелый (>12 мес),64513.0,2.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218663,Zein,Zein,хром/серебро/сталь/серый,2023-06-01,Смесители,Для кухни,NaN,"Смеситель для кухни ZEIN ZC2034, высокий корпу...",6110,2,...,не определен,0,C,0,5631.260014,0.543,24,зрелый (>12 мес),19071.0,13.6
218664,Dokorona,Dokorona,0,2024-01-01,Смесители,Для кухни,NaN,Смеситель кухонный с гибким изливом (гусаком) ...,2893,1,...,не определен,0,C,0,5859.589131,0.494,21,зрелый (>12 мес),7707.0,4.4
218665,Нет бренда,Нет бренда,черный,2023-05-01,Смесители,Для кухни,NaN,"Смеситель для кухни, Кран однорычажный стально...",1194,1,...,однорычажный,0,A,0,5563.708806,0.215,24,зрелый (>12 мес),145531.0,50.7
218666,Нет бренда,Нет бренда,черный,2023-06-01,Смесители,Для кухни,NaN,"Смеситель для кухни, Кран однорычажный стально...",1194,1,...,однорычажный,0,A,0,5631.260014,0.212,24,зрелый (>12 мес),145531.0,50.7
